In [45]:
import pandas as pd

df = pd.read_csv("DelayedFlights.csv")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1936758 entries, 0 to 1936757
Data columns (total 30 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Unnamed: 0         int64  
 1   Year               int64  
 2   Month              int64  
 3   DayofMonth         int64  
 4   DayOfWeek          int64  
 5   DepTime            float64
 6   CRSDepTime         int64  
 7   ArrTime            float64
 8   CRSArrTime         int64  
 9   UniqueCarrier      object 
 10  FlightNum          int64  
 11  TailNum            object 
 12  ActualElapsedTime  float64
 13  CRSElapsedTime     float64
 14  AirTime            float64
 15  ArrDelay           float64
 16  DepDelay           float64
 17  Origin             object 
 18  Dest               object 
 19  Distance           int64  
 20  TaxiIn             float64
 21  TaxiOut            float64
 22  Cancelled          int64  
 23  CancellationCode   object 
 24  Diverted           int64  
 25  CarrierDelay      

## Usunięcie mało popularnych lotów

In [46]:
df["RouteFlight"] = (
    df["UniqueCarrier"].astype(str) + "_" +
    df["FlightNum"].astype(str) + "_" +
    df["Origin"] + "_" +
    df["Dest"]
)

flight_counts = df["RouteFlight"].value_counts()

min_flights = 50

valid_flights = flight_counts[flight_counts >= min_flights].index

df = df[df["RouteFlight"].isin(valid_flights)]


## Obliczenie numeru tygodnia


In [47]:
df["FlightDate"] = pd.to_datetime(
    df["Year"].astype(str) + "-" +
    df["Month"].astype(str) + "-" +
    df["DayofMonth"].astype(str)
)

df["WeekOfYear"] = df["FlightDate"].dt.isocalendar().week

## Wybór tygodni do analizy

In [48]:
weeks = [3, 15, 29, 41]

df_sample = df[df["WeekOfYear"].isin(weeks)]

In [49]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
Index: 71373 entries, 12402 to 1627229
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Unnamed: 0         71373 non-null  int64         
 1   Year               71373 non-null  int64         
 2   Month              71373 non-null  int64         
 3   DayofMonth         71373 non-null  int64         
 4   DayOfWeek          71373 non-null  int64         
 5   DepTime            71373 non-null  float64       
 6   CRSDepTime         71373 non-null  int64         
 7   ArrTime            71181 non-null  float64       
 8   CRSArrTime         71373 non-null  int64         
 9   UniqueCarrier      71373 non-null  object        
 10  FlightNum          71373 non-null  int64         
 11  TailNum            71373 non-null  object        
 12  ActualElapsedTime  71153 non-null  float64       
 13  CRSElapsedTime     71370 non-null  float64       
 14  AirTi

## Usunięcie anulowanych lotów

In [50]:
df_sample = df_sample[df_sample["Cancelled"] == 0]

## Zbudowanie binarnych targetów

In [51]:
df_sample["ArrDelay15"] = (df_sample["ArrDelay"] > 15).astype(int)
df_sample["DepDelay15"] = (df_sample["DepDelay"] > 15).astype(int)
df_sample = df_sample.drop(columns=["ArrDelay", "DepDelay"])

## Zmiana formatu godziny

In [52]:
df_sample["DepHour"] = df_sample["CRSDepTime"] // 100
df_sample["DepMin"] = df_sample["CRSDepTime"] - df_sample["DepHour"] * 100

df_sample["ArrHour"] = df_sample["CRSArrTime"] // 100
df_sample["ArrMin"] = df_sample["CRSArrTime"] - df_sample["ArrHour"] * 100

## Usunięcie danych "z przyszłości"

In [53]:
drop_cols = [
    "DepTime",
    "ArrTime",
    "ActualElapsedTime",
    "AirTime",
    "TaxiIn",
    "TaxiOut",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
    "Unnamed: 0",
    "CancellationCode",
    "RouteFlight",
    "Cancelled",
    "Diverted",
    "CRSDepTime",
    "CRSArrTime"
]

df_sample = df_sample.drop(columns=drop_cols)

In [54]:
df_sample.info()


<class 'pandas.core.frame.DataFrame'>
Index: 71369 entries, 12402 to 1627229
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Year            71369 non-null  int64         
 1   Month           71369 non-null  int64         
 2   DayofMonth      71369 non-null  int64         
 3   DayOfWeek       71369 non-null  int64         
 4   UniqueCarrier   71369 non-null  object        
 5   FlightNum       71369 non-null  int64         
 6   TailNum         71369 non-null  object        
 7   CRSElapsedTime  71366 non-null  float64       
 8   Origin          71369 non-null  object        
 9   Dest            71369 non-null  object        
 10  Distance        71369 non-null  int64         
 11  FlightDate      71369 non-null  datetime64[ns]
 12  WeekOfYear      71369 non-null  UInt32        
 13  ArrDelay15      71369 non-null  int64         
 14  DepDelay15      71369 non-null  int64         
 15  D

In [55]:
df_sample.to_csv("DelayedFlights_Sample.csv", index=False)